# Benchmark Interleaved Splitting

This notebook implements the benchmark using the **Interleaved 4:1:1** splitting strategy. 
It uses the refactored code from the `src` package.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
import matplotlib.pyplot as plt

# Import from src
from src.data_processing import generate_cyclical_features, scale_data_selectively
from src.splitting import get_interleaved_splits
from src.models import TinyTGT, GlobalFitLocalApplySARIMA
from src.evaluation import RollingDataset, prepare_xgb_data, run_evaluation, print_metrics, get_scaling_factors

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
CONFIG = {
    # Date_Range that will be used to generate the temporal features 
    "START_DATE": "2024-01-01",
    "FREQUENCY": "H",                 
    
    # Dataset
    "DATA_PATH": "data_out/no_pertubations/case14_ieee/raw", 
    
    # Features
    "USE_ONLY_LOAD": True,              # Only use active load P
    "BINARY_ADJACENCY": True,           # True = Graph is 0/1 (no admittance)
    
    # Splitting (Interleaved 4:1:1 Strategy)
    "BLOCK_SIZE_HOURS": 168,            # 1 Woche
    "CYCLE_SCHEME": 6,                  #! Function is not truly dynamicly coded
    
    # Forecasting  (Triebe et al. Logic)
    "INPUT_WINDOW": 168,                # Lookback: 7 Tage
    "FORECAST_HORIZON": 33,             # Predict: Rest of Day (9h) + Next Day (24h)
    "TARGET_DAY_START_IDX": 9,          # Target Day" (00:00 tmrw)
    "EVAL_HOUR": 14,                    # forecasting is done always at 2pm    
    # Baselines
    "SNAIVE_LAG": 48                    # Benchmark: value 48h ago
}

In [3]:
# Load Bus and Branch DataFrames
# Adjusted paths relative to the notebook location
bus_df = pd.read_parquet('../data_out/no_pertubations/case14_ieee/raw/bus_data.parquet')
branch_df = pd.read_parquet('../data_out/no_pertubations/case14_ieee/raw/branch_data.parquet')

# Build Edge Index from Branch Data
structure = branch_df[branch_df['load_scenario_idx'] == 0].sort_values("idx")
edge_index = list(structure[['from_bus', 'to_bus']].itertuples(index=False, name=None))

# print dims
m = len(edge_index)
n=bus_df['bus'].nunique()
print(f"Topology Loaded: n={n} Nodes, m={m} Edges")

# X: Input Loads (Active Power 'Pd')
loads_pivot = bus_df.pivot(index='load_scenario_idx', columns='bus', values='Pd').fillna(0)
# Y: Target Flows (Active Power Flow 'pf')
flows_pivot = branch_df.pivot(index='load_scenario_idx', columns='idx', values='pf').fillna(0)

# Convert to Numpy Arrays
loads_matrix = loads_pivot.values.astype(np.float32)
flows_matrix = flows_pivot.values.astype(np.float32)
print(f"shapes: Loads: {loads_matrix.shape}, Flows: {flows_matrix.shape}")

# Adjacency mask (neighbors + self)
A = np.zeros((n,n), dtype=bool)
for (i,j) in edge_index:
    A[i,j]=True; A[j,i]=True
for i in range(n): A[i,i]=True
A_mask = torch.from_numpy(A).to(device)

Topology Loaded: n=14 Nodes, m=20 Edges
shapes: Loads: (8760, 14), Flows: (8760, 20)


In [4]:
# --- Prepare Input Tensor X --- 
# 1. gen Time Features 
T = loads_matrix.shape[0]
time_features_global = generate_cyclical_features(T, CONFIG["START_DATE"], CONFIG["FREQUENCY"]) # [T, 6]
print(f"Time Features generated. Shape: {time_features_global.shape}")

# 2. gen input tensor X
load_tensor = loads_matrix[:, :, np.newaxis]  # [T, N] -> [T, N, 1]
time_tensor_expanded = np.broadcast_to(
    time_features_global[:, np.newaxis, :], 
    (time_features_global.shape[0], loads_matrix.shape[1], time_features_global.shape[1])
) # [T, 6] -> [T, N, 6]  (*read-only view)
X = np.concatenate([load_tensor, time_tensor_expanded], axis=2) # [T, N, 1] + [T, N, 6] -> [T, N, 7]
print(f"Final Input Tensor Shape: {X.shape} (Time, Buses, Features)")

# --- Split Data ---
train_idx, val_idx, test_idx = get_interleaved_splits(T, CONFIG)
print(f"Train Hours: {len(train_idx)}")
print(f"Val Hours:   {len(val_idx)}")
print(f"Test Hours:  {len(test_idx)}")

# --- 3. Scaling (Selective) ---
scaled_tensor, scaler = scale_data_selectively(X, train_idx)
print(f"Scaled Mean (Load): {scaled_tensor[:,:,0].mean():.4f}")
print(f"Scaled Mean (Hour_Sin): {scaled_tensor[:,:,1].mean():.4f} (Should be near 0 but unscaled)")

Time Features generated. Shape: (8760, 6)
Final Input Tensor Shape: (8760, 14, 7) (Time, Buses, Features)
Train Hours: 6048
Val Hours:   1344
Test Hours:  1344
Scaled Mean (Load): -0.0003
Scaled Mean (Hour_Sin): 0.0000 (Should be near 0 but unscaled)


f:\studium\Thesis_Repo\phase1_tgt_model\src\data_processing.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start=start_date, periods=T, freq=frequency)


In [5]:
# --- 4. Create Datasets & Loaders ---
train_dataset = RollingDataset(scaled_tensor, train_idx, CONFIG["INPUT_WINDOW"], CONFIG["FORECAST_HORIZON"])
val_dataset   = RollingDataset(scaled_tensor, val_idx,   CONFIG["INPUT_WINDOW"], CONFIG["FORECAST_HORIZON"])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)

# --- 5. Test One Batch ---
x_batch, y_batch = next(iter(train_loader))
print(f"\nBatch Shapes:")
print(f"X (Input):  {x_batch.shape}  -> [Batch, 168, 14, 7]")
print(f"Y (Target): {y_batch.shape}   -> [Batch, 33, 14, 7]")


Batch Shapes:
X (Input):  torch.Size([4, 168, 14, 7])  -> [Batch, 168, 14, 7]
Y (Target): torch.Size([4, 33, 14, 7])   -> [Batch, 33, 14, 7]


In [7]:
# --- XGBoost Training ---
# 1. Prepare Data mit Downsampling
X_train_xgb, Y_train_xgb = prepare_xgb_data(
    scaled_tensor, 
    train_idx, 
    CONFIG["INPUT_WINDOW"], 
    CONFIG["FORECAST_HORIZON"],
    step_size=1  # <--- WICHTIG: RAM sparen
)

print(f"XGB Train Shape: {X_train_xgb.shape}")

# 2. Setup GPU XGBoost
xgb_estimator = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    objective='reg:squarederror',
    # GPU Konfiguration:
    device='cuda',       
    tree_method='hist',
    n_jobs=4
)

# 3. Wrapper
xgb_model = MultiOutputRegressor(xgb_estimator, n_jobs=1)

# 4. Fit
print("Training Global XGBoost on GTX 970...")
xgb_model.fit(X_train_xgb, Y_train_xgb) # type: ignore
print("XGBoost Training Complete.")

Allocating RAM for 82208 samples...


Building XGB Dataset: 100%|██████████| 5872/5872 [00:00<00:00, 16000.15it/s]


XGB Train Shape: (82208, 175)
Training Global XGBoost on GTX 970...
XGBoost Training Complete.


In [8]:
# --- TinyTGT Training ---

# Setup
import gc
if 'model' in locals(): del model
if 'opt' in locals(): del opt
gc.collect()
torch.cuda.empty_cache() 

# Model
model = TinyTGT(n_nodes=14, d_model=64, n_heads=4, in_feat=7, out_steps=33).to(device)

opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
epochs = 15 

print(f"Restarting TinyTGT Training (Batch Size {4})...")

for ep in range(1, epochs+1):
    model.train()
    train_loss = 0.0
    batch_count = 0
    
    for xb, yb in train_loader:
        xb = xb.float().to(device) 
        yb = yb.float().to(device) 
        
        opt.zero_grad()
        yhat = model(xb, A_mask) 
        
        y_true = yb[..., 0] # Target Slicing
        
        loss = loss_fn(yhat, y_true)
        loss.backward()
        opt.step()
        
        train_loss += loss.item()
        batch_count += 1
    
    avg_train_loss = train_loss / batch_count
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.float().to(device)
            y_true = yb[..., 0].float().to(device)
            
            yhat = model(xb, A_mask)
            val_loss += loss_fn(yhat, y_true).item()
            val_batches += 1
            
    print(f"Epoch {ep}: Train MSE={avg_train_loss:.6f} | Val MSE={val_loss/val_batches:.6f}")

Restarting TinyTGT Training (Batch Size 4)...
Epoch 1: Train MSE=0.034565 | Val MSE=0.014622
Epoch 2: Train MSE=0.010983 | Val MSE=0.010101
Epoch 3: Train MSE=0.008432 | Val MSE=0.015610
Epoch 4: Train MSE=0.007663 | Val MSE=0.011885
Epoch 5: Train MSE=0.007123 | Val MSE=0.009316
Epoch 6: Train MSE=0.006415 | Val MSE=0.009036
Epoch 7: Train MSE=0.005845 | Val MSE=0.008635
Epoch 8: Train MSE=0.005700 | Val MSE=0.009043
Epoch 9: Train MSE=0.005589 | Val MSE=0.008635
Epoch 10: Train MSE=0.005204 | Val MSE=0.008143
Epoch 11: Train MSE=0.004724 | Val MSE=0.009095
Epoch 12: Train MSE=0.004434 | Val MSE=0.008709
Epoch 13: Train MSE=0.004444 | Val MSE=0.008645
Epoch 14: Train MSE=0.004069 | Val MSE=0.008797
Epoch 15: Train MSE=0.003587 | Val MSE=0.008747


In [ ]:
# --- SARIMA Global Fit ---
sarima_model = GlobalFitLocalApplySARIMA(order=(2, 0, 0), seasonal_order=(1, 0, 1, 24))
sarima_model.fit(scaled_tensor, train_idx, 672)


Fitting SARIMA Parameters per Bus (Local Training)...


Fitting Bus Models: 100%|██████████| 14/14 [00:30<00:00,  2.19s/it]


In [10]:
# --- Evaluation ---
# 1. Scaling Factors berechnen (auf Train Data!)
denom_mae, denom_mse = get_scaling_factors(scaled_tensor, train_idx, scaler, m=24)
print(f"Scaling Baseline (Train): MAE={denom_mae:.4f}, MSE={denom_mse:.4f}")

# 2. Evaluation laufen lassen
results = run_evaluation(scaled_tensor, test_idx, xgb_model, model, sarima_model, A_mask, scaler, CONFIG, device=device)

# 3. Metriken drucken
print_metrics(results, denom_mae, denom_mse)

Scaling Baseline (Train): MAE=0.6719, MSE=2.2572
Evaluating 56 days...


  0%|          | 0/56 [00:00<?, ?it/s]f:\studium\Thesis_Repo\.venv\Lib\site-packages\xgboost\core.py:774: UserWarning: [11:54:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
100%|██████████| 56/56 [00:21<00:00,  2.61it/s]


=== FINAL BENCHMARK RESULTS ===
Model      | RMSE       | MAE        | MASE       | MSSE      
--------------------------------------------------------------
SNAIVE     | 1.6462     | 0.8188     | 1.2187     | 1.2006
XGB        | 1.1115     | 0.5950     | 0.8856     | 0.5473
SARIMA     | 1.6849     | 0.8776     | 1.3062     | 1.2577
TinyTGT    | 1.2750     | 0.7611     | 1.1328     | 0.7202
--------------------------------------------------------------
🏆 WINNER (RMSE): XGB
